In [1]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen3.5-9b",
    trust_remote_code=True,
)


/workspace/VLM2Vec/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.append('/workspace/VLM2Vec')

from src.arguments import ModelArguments, DataArguments
from src.model.model import MMEBModel
from src.model.processor import load_processor, QWEN3_5, VLM_IMAGE_TOKENS, Qwen3_5_process_fn
from src.utils.basic_utils import batch_to_device
from PIL import Image
import torch

model_args = ModelArguments(
    model_name='Qwen/Qwen3.5-9b',
    # checkpoint_path='TIGER-Lab/VLM2Vec-Qwen3.5-9b',
    pooling='last',
    normalize=True,
    model_backbone='qwen3_5',
    # lora=True
)
data_args = DataArguments()

processor = load_processor(model_args, data_args)
model = MMEBModel.load(model_args)
# model = model.to('cuda', dtype=torch.bfloat16)
# model.eval()

[2026-05-16 08:33:24,434] INFO [src.utils.basic_utils:21] Loading processor from: Qwen/Qwen3.5-9b
[2026-05-16 08:33:24,436] DEBUG [httpcore.connection:47] close.started
[2026-05-16 08:33:24,437] DEBUG [httpcore.connection:47] close.complete
[2026-05-16 08:33:24,437] DEBUG [httpcore.connection:47] connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=10 socket_options=None
[2026-05-16 08:33:24,444] DEBUG [httpcore.connection:47] connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x74a58741df70>
[2026-05-16 08:33:24,445] DEBUG [httpcore.connection:47] start_tls.started ssl_context=<ssl.SSLContext object at 0x74a9c2165250> server_hostname='huggingface.co' timeout=10
[2026-05-16 08:33:24,451] DEBUG [httpcore.connection:47] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x74a5866eb080>
[2026-05-16 08:33:24,451] DEBUG [httpcore.http11:47] send_request_headers.started request=<Request [b'HEAD']>
[2026-05-16 08

[2026-05-16 08:33:24,557] DEBUG [httpcore.http11:47] receive_response_headers.complete return_value=(b'HTTP/1.1', 307, b'Temporary Redirect', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'86'), (b'Connection', b'keep-alive'), (b'Date', b'Sat, 16 May 2026 08:33:24 GMT'), (b'Location', b'/Qwen/Qwen3.5-9B/resolve/main/processor_config.json'), (b'X-Powered-By', b'huggingface-moon'), (b'X-Request-Id', b'Root=1-6a082bd4-7f5950430e0658ff4be662cd;8bcb1021-ee13-480b-bd8e-6c590e619ba0'), (b'RateLimit', b'"resolvers";r=2915;t=246'), (b'RateLimit-Policy', b'"fixed window";"resolvers";q=3000;w=300'), (b'cross-origin-opener-policy', b'same-origin'), (b'Referrer-Policy', b'strict-origin-when-cross-origin'), (b'Access-Control-Max-Age', b'86400'), (b'Access-Control-Allow-Origin', b'https://huggingface.co'), (b'Vary', b'Origin, Accept'), (b'Access-Control-Expose-Headers', b'X-Repo-Commit,X-Request-Id,X-Error-Code,X-Error-Message,X-Total-Count,ETag,Link,Accept-Ranges,Content-Ran

In [3]:
# Image + Text -> Text
inputs = processor(text=f'{VLM_IMAGE_TOKENS[QWEN3_5]} Represent the given image with the following question: What is in the image',
                   images=Image.open('/workspace/VLM2Vec/assets/example.jpg'),
                   return_tensors="pt")
inputs = {key: value.to('cuda') for key, value in inputs.items()}
inputs['pixel_values'] = inputs['pixel_values'].unsqueeze(0)
inputs['image_grid_thw'] = inputs['image_grid_thw'].unsqueeze(0)

[2026-05-16 08:33:40,420] DEBUG [PIL.Image:421] Importing JpegImagePlugin


In [4]:
inputs

{'input_ids': tensor([[248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
          248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056, 248056,
   

In [41]:
processor_inputs = dict(text=[f'{VLM_IMAGE_TOKENS[QWEN3_5]} Represent the given image with the following question: What is in the image',"Encode this text","encode this text"],
                   images=[[Image.open('/workspace/VLM2Vec/assets/example.jpg')],None,None],
                   return_tensors="pt")

inputs = Qwen3_5_process_fn(
    processor_inputs,
    processor)

In [43]:
inputs['pixel_values'][0].shape

(1900, 1536)

In [42]:
inputs['image_grid_thw']


[array([[ 1, 38, 50]]), None, None]

In [21]:
import torch
import numpy as np

# print("Pixel values shape:", torch.tensor(np.array(inputs['pixel_values'])).shape)
# print("Image grid shape:", torch.tensor(np.array(inputs['image_grid_thw'])).shape)
# print("pixel_values_videos shape:", torch.tensor(np.array(inputs['pixel_values_videos'])).sha
# 
# 

inputs['video_grid_thw']


[None]

In [39]:
# get size of the image
Image.open('/workspace/VLM2Vec/assets/example.jpg').size

(800, 599)